In [ ]:
import sys
print("Path agregado")

In [ ]:
import os
import pandas as pd
import ee
import geemap
from datetime import datetime, timedelta
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union

# Autenticación e inicialización
ee.Authenticate()
ee.Initialize(project='TU-PROYECTO-GEE')

# Configuración
satelite = 'LANDSAT/LC08/C02/T1_L2'  # Landsat 9
interDate = '2026-07-31'  # Ajusta esta fecha según disponibilidad de Landsat 8
value_cl = 30  # Máximo porcentaje de nubes
shapefile_path = r"C:\ruta\a\tu\area_de_interes.shp"
buffer_distance = 5000  # Buffer en metros

def shapefile_to_ee_polygon(shapefile_path, buffer_distance=1000):
    try:
        gdf = gpd.read_file(shapefile_path)
        if gdf.crs is None:
            raise ValueError("El shapefile no tiene un sistema de coordenadas definido.")
        gdf = gdf.to_crs('EPSG:32719')
        gdf['geometry'] = gdf.buffer(buffer_distance)
        gdf = gdf.to_crs('EPSG:4326')
        polygon = unary_union(gdf.geometry)
        if isinstance(polygon, Polygon):
            coords = list(polygon.exterior.coords)
            return ee.Geometry.Polygon([coords])
        elif isinstance(polygon, MultiPolygon):
            multipolygon_coords = []
            for part in polygon.geoms:
                multipolygon_coords.append([list(part.exterior.coords)])
            return ee.Geometry.MultiPolygon(multipolygon_coords)
        else:
            raise ValueError("La geometría resultante no es válida.")
    except Exception as e:
        print(f"Error al procesar el shapefile: {e}")
        return None

AOI = shapefile_to_ee_polygon(shapefile_path, buffer_distance)

if AOI:
    interDate = datetime.strptime(interDate, '%Y-%m-%d')
    startDate = interDate - timedelta(days=365)
    endDate = interDate + timedelta(days=1)
    print(f"Buscando imágenes entre {startDate} y {endDate}...")

    Collection_L8 = (
        ee.ImageCollection(satelite)
        .filterDate(startDate.strftime('%Y-%m-%d'), endDate.strftime('%Y-%m-%d'))
        .filterBounds(AOI)
        .filter(ee.Filter.lt('CLOUD_COVER_LAND', value_cl))
    )

    n = Collection_L8.size().getInfo()
    if n == 0:
        raise ValueError("No se encontraron imágenes Landsat 8 para ese período.")
    else:
        print(f'Total de imágenes encontradas: {n}')
        image_info = Collection_L8.aggregate_array('system:time_start').getInfo()
        cloud_coverage = Collection_L8.aggregate_array('CLOUD_COVER_LAND').getInfo()

        df = pd.DataFrame({
            'Fecha': [datetime.utcfromtimestamp(date / 1000).strftime('%Y-%m-%d') for date in image_info],
            'Porcentaje': cloud_coverage
        })

        df_agrupado = df.groupby('Fecha', as_index=False)['Porcentaje'].mean()
        print(df_agrupado)

        # Crear el mapa
        Map = geemap.Map()

        for fecha in df_agrupado['Fecha']:
            next_day = (datetime.strptime(fecha, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')
            Collection_fecha = (
                ee.ImageCollection(satelite)
                .filterDate(fecha, next_day)
                .filterBounds(AOI)
            )

            # Calcular mediana y aplicar escala
            medianpixels = Collection_fecha.median().clip(AOI).multiply(0.0000275).add(-0.2)

            # Reducir para obtener valores estadísticos
            stats = medianpixels.reduceRegion(
                reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
                geometry=AOI,
                scale=30,
                maxPixels=1e13
            ).getInfo()

            try:
                bands = ['SR_B4', 'SR_B3', 'SR_B2']
                if stats and all(f'{band}_mean' in stats and f'{band}_stdDev' in stats for band in bands):
                    vis_params = {
                        'bands': bands,
                        'min': min(stats[f'{band}_mean'] - stats[f'{band}_stdDev'] for band in bands),
                        'max': max(stats[f'{band}_mean'] + stats[f'{band}_stdDev'] for band in bands),
                        'gamma': 1.8
                    }
                    Map.addLayer(medianpixels, vis_params, f"Landsat8_{fecha}")
                    print(f"Agregada imagen de {fecha} al mapa.")
                else:
                    print(f"Datos estadísticos no disponibles para {fecha}.")
            except Exception as e:
                print(f"No se pudo visualizar la imagen de {fecha}: {e}")

        # Exportar como HTML
        html_path = "mapa_landsat8.html"
        Map.to_html(html_path)
        print(f"Visualización completada. Mapa exportado a: {html_path}")
